In [1]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
import inspect
import json
import logging
import matplotlib.pyplot as plt
import mlflow
import nltk
from nltk.corpus import stopwords
import numpy as np
import os
from pathlib import Path
import pandas as pd
import random
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_curve
import shutil
import torch
from torch import nn, optim
from torch.amp import GradScaler, autocast
from torch.cuda import Event
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, CosineAnnealingLR, OneCycleLR, ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, AUROC, AveragePrecision, PrecisionRecallCurve
from tqdm.notebook import tqdm
import warnings
import zipfile
warnings.filterwarnings("ignore", message=".*Pickle or CloudPickle.*")
nltk.download('stopwords', quiet=True)

[nltk_data] Error loading stopwords: <urlopen error [WinError 10060] A
[nltk_data]     connection attempt failed because the connected party
[nltk_data]     did not properly respond after a period of time, or
[nltk_data]     established connection failed because connected host
[nltk_data]     has failed to respond>


False

In [2]:
%load_ext requirements

In [3]:
class SystemConfig:
    IS_DETERMINISTIC = False
    _PROJECT_ROOT = Path.cwd()  # research/ folder
    _DB_PATH = _PROJECT_ROOT / "mlflow-quora-questions-pairs.db"
    MLFLOW_TRACKING_URI = f"sqlite:///{_DB_PATH.as_posix()}"
    NEXT_LINE_COUNTER = 180
    SEED = 28
    USED_SCALER = False
    NEXT_LINE_COUNTER = 180
    @staticmethod
    def get_device():
        '''
        Detects the best available device for PyTorch.
        Priority: TPU -> GPU (CUDA) -> CPU
        '''
        # 1. Check for TPU (requires torch_xla)
        try:
            import torch_xla.core.xla_model as xm
            device = xm.xla_device()
            print(f">>> Using TPU: {device}")
        except ImportError:
            # 2. Check for GPU (CUDA)
            if torch.cuda.is_available():
                device = torch.device("cuda")
                print(f">>> Using GPU: {torch.cuda.get_device_name(0)}")
            # 3. Fallback to CPU
            else:
                device = torch.device("cpu")
                print(">>> Using CPU")
                
        return device
    DEVICE = get_device.__func__()

    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

>>> Using GPU: NVIDIA GeForce GTX 1650


In [4]:
class PathConfig:
    ROOT_DIR = Path().cwd()
    EMB_DIR = ROOT_DIR.parent.parent
    EMB_PATH = EMB_DIR / "glove.6B.100d.txt"
    ARTIFACT_DIR = ROOT_DIR / "artifacts"
    REQ_TEXT = ROOT_DIR / "requirements.txt"
    REQ_SCRIPT = ROOT_DIR / "requirements.py"
    MODEL_SCRIPT = ROOT_DIR / "model_architecture.py"
    NOTEBOOK_PATH = ROOT_DIR / "LSTM-attention-train.ipynb"
    DATA_DIR = ROOT_DIR / "data"
    MLFLOW_DIR = ROOT_DIR / "mlruns"
    CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoint"
    CONFIG_PATH = ARTIFACT_DIR / "configs.json"
    HISTORY_PATH = ARTIFACT_DIR / "training_history.json"
    VOCABS_PATH = ARTIFACT_DIR / "vocabs.json"
    LABEL_MAPPING_PATH = ARTIFACT_DIR / "label_mapping.json"
    TRAIN_CSV_PATH = DATA_DIR / "train.csv"
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

    @classmethod
    def update_requirements(cls):
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython:
            ipython.run_line_magic('updatereqs', str(cls.NOTEBOOK_PATH))

In [5]:
class TokenConfig:
    PAD_TOKEN = '<PAD>'
    UNK_TOKEN = '<UNK>'
    PAD_IDX = 0
    UNK_IDX = 1
    LOWERCASE = True
    UPPERCASE = False
    MAX_LENGTH = 50
    VOCAB_SIZE = 20000
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [6]:
class LoaderConfig:
    BATCH_SIZE = 128
    NUM_WORKERS = 0
    IS_PIN_MEMORY = True
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [7]:
class ModelConfig:
    MODEL_TYPE = "LSTM_attention"
    ATTENTION_TYPE = "ESIM_Style-MultiHead-CrossAttention-Bahdanau"

    NUM_HEADS = 4                                     
    SELF_ATTENTION_NUM_HEADS = NUM_HEADS
    CROSS_ATTENTION_NUM_HEADS = NUM_HEADS

    ATTENTION_DROPOUT = 0.0
    SKIP_CONNECTION_IN_ATTENTION = True
    POOLING_TYPE = "Average and Max"
    
    # Embedding
    EMB_NORM = False
    FREEZE_TOKEN_EMBEDDING = True
    TOKEN_EMBEDDING = "gloVe-6B-100d"
    EMB_DIM = 100
    EMB_DP = 0.0

    # Model
    LOSS = "BCE with Logits"
    ENHC_LSTM_DROPOUT = 0.4
    ENHC_LSTM_NUM_LAYERS = 2
    ENHC_LSTM_DIM = 256
    ENHC_LSTM_BIDIRECTIONAL = True
    ENHC_LSTM_OUT = ENHC_LSTM_DIM * (2 if ENHC_LSTM_BIDIRECTIONAL else 1)
    ENHC_LSTM_NORM = False
    COMP_LSTM_DIM = 128
    COMP_LSTM_BIDIRECTIONAL = True
    COMP_LSTM_OUT = COMP_LSTM_DIM * (2 if COMP_LSTM_BIDIRECTIONAL else 1)
    COMP_LSTM_NORM = False
    COMP_LSTM_DROPOUT = 0.3
    COMP_LSTM_NUM_LAYERS = 2
    PROJ_DIMS = [256]
    PROJ_DROPOUT = 0.3

    # Loss specific settings
    if LOSS == "Contrastive Loss":
        MARGIN = 1.0
    elif LOSS == "BCE with Logits":
        LABEL_SMOOTHING = 0.05
        FC_DIMS = [512, 256]
        FC_DP = 0.5

    MASK_FILL_NUM = -1e10

    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)
            and not isinstance(v, (classmethod, staticmethod))
        }

In [8]:
class TrainConfig:
    LOSS = "BCE with Logits"
    CLIP_NORM = 1.5
    EARLY_STOP_METRIC = "loss"
    SCHEDULER_METRIC = "loss"
    CHECKPOINT_METRIC = "F1Score"
    if CHECKPOINT_METRIC == "loss":
        CHECKPOINT_MODE = "min"
    else:
        CHECKPOINT_MODE = "max"
    EARLY_STOP_MIN_DELTA = 1e-4
    if EARLY_STOP_METRIC in ["F1Score", "Accuracy", "Precision", "Recall"]:
        EARLY_STOP_MODE = "max"
    elif EARLY_STOP_METRIC == "loss":
        EARLY_STOP_MODE = "min"
    EARLY_STOP_PATIENCE = 3
    EPOCHS = 50
    LEARNING_RATE = 3e-4
    METRICS_THRESHOLD = 0.5
    SCHEDULER_TYPE = "CosineAnnealing"
    if SCHEDULER_TYPE == "ReduceLROnPlateau":
        SCHEDULER_FACTOR = 0.5
        SCHEDULER_MIN_LR = 1e-7
        SCHEDULER_PATIENCE = 2
        SCHEDULER_THRESHOLD = 0.01
        SCHEDULER_THRESHOLD_MODE = "rel"
        if SCHEDULER_METRIC in ["F1Score", "Accuracy", "Precision", "Recall"]:
            SCHEDULER_MODE = "max"
        elif SCHEDULER_METRIC == "loss":
            SCHEDULER_MODE = "min"
    elif SCHEDULER_TYPE == "CosineAnnealing":
        SCHEDULER_ETA_MIN = 5e-7
    elif SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        SCHEDULER_T_0 = 5
        SCHEDULER_T_MUT = 2
        SCHEDULER_ETA_MIN = 1e-6
    elif SCHEDULER_TYPE == "OneCycleLR":
        SCHEDULER_PCT_START = 0.3
        SCHEDULER_DIV_FACTOR = 25
        SCHEDULER_FINAL_DIV_FACTOR = 1000

    TRAIN_TEST_SPLIT = 0.9
    UNFREEZE_EPOCH = 3
    WEIGHT_DECAY = 5e-3
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [9]:
class PostProcessingConfig:
    INFERENCE_THRESHOLD = 0.5
    METRICS_THRESHOLD = 0.5
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [10]:
for path in [PathConfig.EMB_PATH, PathConfig.TRAIN_CSV_PATH]:
    assert os.path.exists(path), f"File not found: {path}"

In [11]:
def seed_everything(seed, deterministic=False):
    '''
    Ensures absolute reproducibility.
    '''
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    if deterministic:
        # Only use these for the final "Gold" run to ensure exact results
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(">>> Using STRICT Deterministic mode (Slower).")
    else:
        # Benchmark=True allows cuDNN to find the fastest kernels for your hardware
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        print(">>> Using PROTOTYPING mode (Faster).")
    print(f">>> For Reproducibility, Everything seeded with {seed}!")



In [12]:
def set_scaler(config=SystemConfig):
    if config.USED_SCALER:
        scaler = GradScaler(device=sys_cfg.DEVICE.type, enabled=(sys_cfg.DEVICE.type == 'cuda'))
        print(">>> Scaler used in training")
        return scaler
    else:
        scaler = None
        print(">>> Scaler is not Used")

In [13]:
def clean_artifact_directory(artifact_dir: Path):
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)
        print(f">>> Cleaned local artifact directory: {artifact_dir}")
    artifact_dir.mkdir(parents=True, exist_ok=True)

In [14]:
sys_cfg = SystemConfig()
path_cfg = PathConfig()
token_cfg = TokenConfig()
train_cfg = TrainConfig()
loader_cfg = LoaderConfig()
model_cfg = ModelConfig()
postprc_cfg = PostProcessingConfig()
print(">>> All Configs are set successfully!")
path_cfg.update_requirements()
scaler = set_scaler(sys_cfg)
seed_everything(sys_cfg.SEED, deterministic=sys_cfg.IS_DETERMINISTIC)
print(f">>> Training on: {sys_cfg.DEVICE} with seed = {sys_cfg.SEED}")

>>> All Configs are set successfully!
>>> Scanning: LSTM-attention-train.ipynb
>>> Detected third-party imports: ['IPython', 'matplotlib', 'mlflow', 'nltk', 'numpy', 'pandas', 'sklearn', 'torch', 'torch_xla', 'torchmetrics', 'tqdm']
>>> Skipping 'torch_xla' (no matching package found)
>>> Total packages in requirements.txt: 11
>>> Scaler is not Used
>>> Using PROTOTYPING mode (Faster).
>>> For Reproducibility, Everything seeded with 28!
>>> Training on: cuda with seed = 28


In [15]:
clean_artifact_directory(path_cfg.ARTIFACT_DIR)

>>> Cleaned local artifact directory: C:\Users\98922\Documents\python_scripts\AI\research-notebooks\Quora Questions Pairs\research\artifacts


In [16]:
def get_serializable_configs(configs_dict):
    """Create a JSON-serializable version of configs"""
    serializable = {}
    for section, params in configs_dict.items():
        if section not in ["system", "path"]:
            serializable[section] = params
            continue
            
        serializable[section] = {}
        
        for key, value in params.items():
            if isinstance(value, torch.device):
                # Convert torch.device to string
                serializable[section][key] = str(value)
            elif isinstance(value, Path):
                # Convert Path to string
                serializable[section][key] = str(value)
            elif key == "device":
                # Skip or convert device
                serializable[section][key] = str(value) if value.type == "cuda" else "cpu"
            else:
                serializable[section][key] = value
    
    return serializable

In [17]:
def configs_dict(config_path):
    configs = {}
    configs_names = [
        SystemConfig, PathConfig, TokenConfig, LoaderConfig, TrainConfig, ModelConfig
    ]
    for config in configs_names:
        cfg_clean = f"{config.__name__.replace("Config", "").lower()}"
        configs[cfg_clean] = config.to_dict()

    serializable_configs = get_serializable_configs(configs)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(serializable_configs, f, ensure_ascii=False, indent=2)
    print(">>> Configs Saved as JSON File!")
    return configs

configs = configs_dict(path_cfg.CONFIG_PATH)

>>> Configs Saved as JSON File!


In [18]:
df = pd.read_csv(path_cfg.TRAIN_CSV_PATH)
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [19]:
print(f"Number of all pairs in research: {len(df)}")

Number of all pairs in research: 404290


In [20]:
df.isna().sum()

id              0
qid1            0
qid2            0
question1       1
question2       2
is_duplicate    0
dtype: int64

In [21]:
df["is_duplicate"].value_counts().sort_index()

is_duplicate
0    255027
1    149263
Name: count, dtype: int64

In [22]:
not_dupl = (df["is_duplicate"] == 0).sum()
is_dupl = (df["is_duplicate"] == 1).sum()
total = len(df)
print(
    f"Is duplicate in % : {is_dupl / total * 100}\n"
    f"not duplicate in % : {not_dupl / total * 100}"
)

Is duplicate in % : 36.9197853026293
not duplicate in % : 63.08021469737069


In [23]:
class QuoraPreproccesor:
    def __init__(self, config=token_cfg):
        self.config = config
    def _clean_text(self, text):
        if self.config.LOWERCASE:
            text = text.lower()
        if self.config.UPPERCASE:
            text = text.upper()
    
        text = re.sub(r'!', ' ! ', text)
        text = re.sub(r'\?', ' ? ', text)
        text = re.sub(r'\？', ' ? ', text)
        
        text = re.sub(r'\.', ' . ', text)
        text = re.sub(r',', ' , ', text)
        text = re.sub(r'-', ' - ', text)
        text = re.sub(r'\–', ' - ', text)
        text = re.sub(r'\(', ' ( ', text)
        text = re.sub(r'\)', ' ) ', text)
        text = re.sub(r'\"', " \' ", text)
        text = re.sub(r'\\', " \\ ", text)
        text = re.sub(r'/', " / ", text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    def preprocess_df(self, df):
        df = df.copy()
        df['question1'] = df['question1'].fillna('')
        df['question2'] = df['question2'].fillna('')
        df['question1'] = df['question1'].apply(self._clean_text)
        df['question2'] = df['question2'].apply(self._clean_text)
        print(">>> Preprocessing complete!")
        return df

In [24]:
preprocessor = QuoraPreproccesor(config=token_cfg)
df = preprocessor.preprocess_df(df)

>>> Preprocessing complete!


In [25]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,1,3,4,what is the story of kohinoor ( koh - i - noor...,what would happen if the indian government sto...,0
2,2,5,6,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,3,7,8,why am i mentally very lonely ? how can i solv...,find the remainder when [math]23^{24}[ / math]...,0
4,4,9,10,"which one dissolve in water quikly sugar , sal...",which fish would survive in salt water ?,0


In [26]:
def show_frequent_tokens(df, top_k=100):
    """Print top‑k most frequent tokens across all questions."""
    nltk_stops = set(stopwords.words('english'))

    counter = Counter()
    total_questions = 0
    for q in df['question1'].fillna('').apply(str):
        counter.update(q.split())
        total_questions += 1
    for q in df['question2'].fillna('').apply(str):
        counter.update(q.split())
        total_questions += 1

    print(f"Total tokens counted: {sum(counter.values())}  |  Unique tokens: {len(counter)}")
    print(f"Questions: {total_questions}")
    print(f"{'Token':<15} {'Count':>7} {'Freq (%)':>9} {'NLTK stop?'}")
    print("-" * 45)
    for word, count in counter.most_common(top_k):
        freq = 100.0 * count / total_questions
        is_stop = word in nltk_stops
        print(f"{word:<15} {count:>7} {freq:>8.2f}% {'yes' if is_stop else 'no'}")

In [84]:
show_frequent_tokens(df)

Total tokens counted: 10200452  |  Unique tokens: 103212
Questions: 808580
Token             Count  Freq (%) NLTK stop?
---------------------------------------------
?                852538   105.44% no
the              377503    46.69% yes
what             311757    38.56% yes
is               269887    33.38% yes
how              220706    27.30% yes
i                215237    26.62% yes
a                211711    26.18% yes
to               205773    25.45% yes
in               197220    24.39% yes
do               161024    19.91% yes
of               159855    19.77% yes
are              145884    18.04% yes
and              133966    16.57% yes
can              114091    14.11% yes
for              104455    12.92% yes
,                101731    12.58% no
you               89505    11.07% yes
why               83960    10.38% yes
.                 74530     9.22% no
my                70943     8.77% yes
best              70522     8.72% no
it                69075     8.54% yes
on

In [27]:
train_df, val_df = train_test_split(
    df, random_state=sys_cfg.SEED, shuffle=True, 
    stratify=df["is_duplicate"], train_size=train_cfg.TRAIN_TEST_SPLIT
)

In [42]:
class QuoraTokenizer:
    def __init__(self, config=token_cfg):
        self.config = config
        self.vocabs = {}
        self.idx2word = {}
        self.mask_words = None
        self.vocab_size = config.VOCAB_SIZE

    def build_vocabs(self, df):
        counter = Counter()
        question1 = df["question1"]
        question2 = df["question2"]
        for q1, q2 in zip(question1, question2):
            counter.update(q1.split())
            counter.update(q2.split())
        most_common = counter.most_common(self.config.VOCAB_SIZE - 2)
        self.vocabs = {
            self.config.PAD_TOKEN: self.config.PAD_IDX,
            self.config.UNK_TOKEN: self.config.UNK_IDX
        }
        for idx, (word, _) in enumerate(most_common, start=2):
            self.vocabs[word] = idx
        self.idx2word = {v: k for k, v in self.vocabs.items()}
        self.vocab_size = len(self.vocabs)
        self._build_stop_mask()
        print(">>> Vocabs created!")

    def _build_stop_mask(self):
        stop_set = set(stopwords.words("english"))
        custom = {
            '?', '!', '.', ',', '-', '...', '..', '/', '\\', '(', ')', '"', "'", '<PAD>', "what's", "<UNK>"
        }
        stop_set.update(custom)

        mask = torch.ones(len(self.vocabs))
        for word, idx in self.vocabs.items():
            if word in stop_set:
                mask[idx] = 0.0
        self.stop_mask = mask

    def encode(self, text):
        tokens = text.split()
        ids = [self.vocabs.get(t, self.vocabs[self.config.UNK_TOKEN]) for t in tokens]
        # truncate
        ids = ids[:self.config.MAX_LENGTH]
        # pad
        ids += [self.vocabs[self.config.PAD_TOKEN]] * (self.config.MAX_LENGTH - len(ids))
        return ids

    def decode(self, ids, remove_pad=True):
        pad_id = self.vocabs[self.config.PAD_TOKEN]
        return " ".join(
            self.idx2word.get(i, self.config.UNK_TOKEN) for i in ids
            if not (remove_pad and i == pad_id)
        )

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "max_length": self.config.MAX_LENGTH,
            "vocab_size": self.vocab_size,
            "special_tokens": {
                "pad": self.config.PAD_TOKEN,
                "unk": self.config.UNK_TOKEN
            },
            "vocabs": self.vocabs,
            "idx2word": {str(k): v for k, v in self.idx2word.items()},  # keys as strings for JSON
            "stop_mask": self.stop_mask.tolist() if self.stop_mask is not None else None
        }
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        print(f">>> Tokenizer saved to {path}")

    def save_label_mapping(self, path):
        path = Path(path)
        label_mapping = {"0": "different", "1": "duplicated"}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(label_mapping, f, ensure_ascii=False, indent=2)
            print(">>> Label mapping saved!")


    def load_embedding(self, emb_dim, emb_dir):
        vocab_size = len(self.idx2word)
        emb_matrix = np.random.normal(0.0, 1.0, (vocab_size, emb_dim))
        with open(emb_dir, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                word = parts[0]
                vector = np.array(parts[1:], dtype=np.float32)
                if word in self.idx2word:
                    idx = self.idx2word[word]
                    emb_matrix[idx] = vector
    
        return torch.FloatTensor(emb_matrix)
        
    @classmethod
    def load(cls, path, config=token_cfg):
        
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Create a new tokenizer using the given config
        tokenizer = cls(config=config)
        tokenizer.vocabs = data['vocabs']
        tokenizer.idx2word = {int(k): v for k, v in data['idx2word'].items()}
        if data['stop_mask'] is not None:
            tokenizer.stop_mask = torch.tensor(data['stop_mask'])
        else:
            tokenizer._build_stop_mask()  # fallback
        print(f">>> Tokenizer loaded from {path}")
        return tokenizer

In [45]:
tokenizer = QuoraTokenizer(config=token_cfg)
tokenizer.build_vocabs(train_df)
vocabs = tokenizer.vocabs
vocab_size = tokenizer.vocab_size
stop_mask = tokenizer.stop_mask
embedding = tokenizer.load_embedding(model_cfg.EMB_DIM, path_cfg.EMB_PATH)
print(">>> Embedding loaded successfully!")
# verify
print(">>> Vocab size:", tokenizer.vocab_size)
print(">>> First 10 vocab entries:", list(tokenizer.vocabs.items())[:10])
tokenizer.save(path_cfg.VOCABS_PATH)
tokenizer.save_label_mapping(path_cfg.LABEL_MAPPING_PATH)

>>> Vocabs created!
>>> Embedding loaded successfully!
>>> Vocab size: 20000
>>> First 10 vocab entries: [('<PAD>', 0), ('<UNK>', 1), ('?', 2), ('the', 3), ('what', 4), ('is', 5), ('how', 6), ('i', 7), ('a', 8), ('to', 9)]
>>> Tokenizer saved to C:\Users\98922\Documents\python_scripts\AI\research-notebooks\Quora Questions Pairs\research\artifacts\vocabs.json
>>> Label mapping saved!


In [46]:
class QuoraDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.q1 = df["question1"].values
        self.q2 = df["question2"].values
        self.label = df["is_duplicate"].values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.q1)

    def pos_class_weight(self, device):
        pos_num = (self.label == 1).sum()
        neg_num = (self.label == 0).sum()
        pos_weight = torch.tensor(neg_num / pos_num).to(device)

        return pos_weight

    def __getitem__(self, idx):
        encoded_q1 = self.tokenizer.encode(self.q1[idx])
        encoded_q2 = self.tokenizer.encode(self.q2[idx])
        label = self.label[idx]

        return {
            "q1": torch.LongTensor(encoded_q1),
            "q2": torch.LongTensor(encoded_q2),
            "label": torch.tensor(label, dtype=torch.float)
        }

In [47]:
train_dataset = QuoraDataset(train_df, tokenizer=tokenizer)
val_dataset = QuoraDataset(val_df, tokenizer=tokenizer)

In [32]:
for i in range(3):
    item = val_dataset[i]
    print(f"\nSample {i}")
    print("Encoded Question 1:")
    print(item["q1"].tolist())
    print(f"\tDecoded text: {tokenizer.decode(item["q1"].tolist(), tokenizer.idx2word)}")
    print("Encoded Question 2:")
    print(item["q2"].tolist())
    print(f"\tDecoded Question 2: {tokenizer.decode(item["q2"].tolist(), tokenizer.idx2word)}")
    print("\t\tLabel:", item["label"])


Sample 0
Encoded Question 1:
[6, 15, 7, 61, 76, 101, 10, 41, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded text: how can i make money online in india ?
Encoded Question 2:
[84, 3, 606, 62, 9, 61, 76, 101, 39, 41, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded Question 2: what's the easiest way to make money online from india ?
		Label: tensor(1.)

Sample 1
Encoded Question 1:
[4, 13, 30, 50, 1820, 55, 8117, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded text: what are some good novels about farmers ?
Encoded Question 2:
[4, 13, 30, 50, 1820, 65, 3, 563, 5, 1093, 396, 8117, 14, 5902, 70, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded Question 2: what are some goo

In [48]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=loader_cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=loader_cfg.NUM_WORKERS,
    pin_memory=loader_cfg.IS_PIN_MEMORY
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=loader_cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=loader_cfg.NUM_WORKERS,
    pin_memory=loader_cfg.IS_PIN_MEMORY
)

In [34]:
batch = next(iter(train_dataloader))

print(f"Batched Input IDs Size: {batch["q1"].shape}")
print(f"Batched Input IDs Size: {batch["q2"].shape}")
print(f"Batched Labels size: {batch["label"].shape}")

Batched Input IDs Size: torch.Size([128, 50])
Batched Input IDs Size: torch.Size([128, 50])
Batched Labels size: torch.Size([128])


In [36]:
unk_words_q1 = sum(
    (i == token_cfg.UNK_IDX) for text in train_df["question1"]
    for i in tokenizer.encode(text)
)
unk_words_q2 = sum(
    (i == token_cfg.UNK_IDX) for text in train_df["question2"]
    for i in tokenizer.encode(text)
)
unk_ratio = (unk_words_q1 + unk_words_q2) / (2*len(train_df["question1"]) * token_cfg.MAX_LENGTH)
print(f"Unknown Words Ratio: {unk_ratio:.4f}")

Unknown Words Ratio: 0.0049


In [49]:
class CrossAttentionHead(nn.Module):
    def __init__(self, hidden_dim, proj_dim, mask_fill_num=model_cfg.MASK_FILL_NUM,
                 dropout=model_cfg.ATTENTION_DROPOUT):
        super().__init__()
        self.W_q = nn.Linear(hidden_dim, proj_dim)
        self.W_k = nn.Linear(hidden_dim, proj_dim)
        self.W_v = nn.Linear(hidden_dim, proj_dim)
        self.V = nn.Linear(proj_dim, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.mask_fill_num = mask_fill_num

    def forward(self, query, key_value, mask_kv):
        Q = self.W_q(query)
        K = self.W_k(key_value)
        V = self.W_v(key_value)
        energy = torch.tanh(Q.unsqueeze(2) + K.unsqueeze(1))
        scores = self.V(energy).squeeze(-1)
        scores = scores.masked_fill(mask_kv.unsqueeze(1) == 0, self.mask_fill_num)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        aligned = torch.bmm(attn_weights, V)
        return aligned

In [50]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads=model_cfg.CROSS_ATTENTION_NUM_HEADS):
        super().__init__()
        head_dim = hidden_dim // num_heads
        self.heads = nn.ModuleList([
            CrossAttentionHead(hidden_dim, head_dim) for _ in range(num_heads)
        ])
        self.out_linear = nn.Linear(head_dim*num_heads, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
    def forward(self, query, key_value, mask_kv):
        x = torch.cat([h(query, key_value, mask_kv) for h in self.heads], dim=-1)
        x = self.out_linear(x)
        return self.norm(query + x)

In [51]:
class MaskedMeanPool(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x, mask):
        lengths = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (x * mask.unsqueeze(-1)).sum(dim=1) / lengths
        return pooled

In [52]:
class SelfAttentivePooling(nn.Module):
    def __init__(self, hidden_dim, proj_dim):
        super().__init__()
        self.attention = SelfAttentionHead(hidden_dim, proj_dim)
        
    def forward(self, x, mask):
        context = self.attention(x, mask)
        return context

In [53]:
class AvgMaxPool(nn.Module):
    def __init__(self, mask_fill_num=model_cfg.MASK_FILL_NUM):
        super().__init__()
        self.mask_fill_num = mask_fill_num

    def forward(self, x, mask):
        lengths = mask.sum(dim=1, keepdim=True).clamp(min=1)
        avg_pool = (x * mask.unsqueeze(-1).sum(dim=1)) / lengths

        x_masked = mask.masked_fill(mask == 0, self.mask_fill_num)
        max_pool, _ = x_masked.max(dim=1)

        return torch.cat([avg_pool, max_pool], dim=-1)

In [226]:
class QuoraSiameseClassifier(nn.Module):
    def __init__(self, vocab_size, config=model_cfg, embedding=None, stop_mask=None):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(vocab_size, config.EMB_DIM)
        self.emb_norm = nn.LayerNorm(config.EMB_DIM) if config.EMB_NORM else nn.Identity()
        self.emb_dropout = nn.Dropout(config.EMB_DP)
        if stop_mask is not None:
            self.register_buffer("stop_mask", stop_mask)
        else:
            self.stop_mask = None
        if embedding is not None:
            print("Glove copied in Embedding Layer...")
            self.embedding.weight.data.copy_(embedding)
            self.embedding.weight.requires_grad = not config.FREEZE_TOKEN_EMBEDDING

        self.enhc_lstm = nn.LSTM(
            input_size=config.EMB_DIM,
            hidden_size=config.ENHC_LSTM_DIM,
            bidirectional=config.ENHC_LSTM_BIDIRECTIONAL,
            num_layers=config.ENHC_LSTM_NUM_LAYERS,
            dropout=config.ENHC_LSTM_DROPOUT if config.ENHC_LSTM_NUM_LAYERS > 1 else 0.0,
            batch_first=True
        )
        self.enhc_lstm_norm = nn.LayerNorm(config.ENHC_LSTM_OUT) if config.ENHC_LSTM_NORM else nn.Identity()
        self.cross_attention = MultiHeadCrossAttention(config.ENHC_LSTM_OUT)
        self.proj = self._build_fc_layers(
            input_dim=4*config.ENHC_LSTM_OUT,
            fc_dims=config.PROJ_DIMS,
            dropout=config.PROJ_DROPOUT
        )
        self.comp_lstm = nn.LSTM(
            input_size=config.PROJ_DIMS[-1],
            hidden_size=config.COMP_LSTM_DIM,
            bidirectional=config.COMP_LSTM_BIDIRECTIONAL,
            num_layers=config.COMP_LSTM_NUM_LAYERS,
            dropout=config.COMP_LSTM_DROPOUT if config.COMP_LSTM_NUM_LAYERS > 1 else 0.0,
            batch_first=True
        )
        self.comp_lstm_norm = nn.LayerNorm(config.COMP_LSTM_OUT) if config.COMP_LSTM_NORM else nn.Identity()
        self.pool = AvgMaxPool(mask_fill_num=model_cfg.MASK_FILL_NUM)
        self.fc_dims = self._build_fc_layers(
            input_dim=4*config.COMP_LSTM_OUT,
            fc_dims=config.FC_DIMS,
            dropout=config.FC_DP
        )
        self.final_layer = nn.Linear(config.FC_DIMS[-1], 1)
        
    def _build_fc_layers(self, input_dim, fc_dims, dropout):
        layers = []
        for dim in fc_dims:
            layers += [
                nn.Linear(input_dim, dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ]
            input_dim = dim
        return nn.Sequential(*layers)
    
    def _create_mask(self, question):
        return (question != 0).float()

    def _encode(self, question):
        emb = self.embedding(question)
        emb = self.emb_norm(emb)
        emb = self.emb_dropout(emb)
        mask = self._create_mask(question)
        if self.stop_mask is not None:
            token_stop_mask = self.stop_mask[question]
            mask = mask * token_stop_mask.float()
        out, _ = self.enhc_lstm(emb)
        out = self.enhc_lstm_norm(out)
        return out, mask

    def forward(self, q1, q2):
        out1, mask1 = self._encode(q1)
        out2, mask2 = self._encode(q2)
        
        cross1 = self.cross_attention(query=out1, key_value=out2, mask_kv=mask2)
        cross2 = self.cross_attention(query=out2, key_value=out1, mask_kv=mask1)

        enhc1 = torch.cat([out1, cross1, out1 - cross1, out1*cross1], dim=-1)
        enhc2 = torch.cat([out2, cross2, out2 - cross2, out2*cross2], dim=-1)

        proj1 = self.proj(enhc1)
        proj2 = self.proj(enhc2)

        comp1, _ = self.comp_lstm(proj1)
        comp2, _ = self.comp_lstm(proj2)
        comp1 = self.comp_lstm_norm(comp1)
        comp2 = self.comp_lstm_norm(comp2)
        
        h1 = self.pool(comp1, mask1)
        h2 = self.pool(comp2, mask2)
        feat = torch.cat([h1, h2], dim=-1)
        final_feat = self.fc_dims(feat)
        logits = self.final_layer(final_feat)
        
        return logits.squeeze(-1)

In [227]:
def export_model_from_notebook(notebook_path: str, output_path: Path,
                               class_names=("AvgMaxPool", "CrossAttentionHead", "MultiHeadCrossAttention",
                                            "QuoraSiameseClassifier", "ModelConfig")):
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Collect all matching cells, but store them by class name to avoid duplicates
    # We'll keep the *last* cell that defines each class.
    class_to_cell = {}
    cell_order = []   # preserve original order for output

    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = cell.get('source', '')
        if isinstance(source, list):
            source = ''.join(source)

        for name in class_names:
            if f"class {name}" in source:
                # Overwrite any previous definition with the same class name
                if name in class_to_cell:
                    # Remove from ordered list to replace later
                    cell_order.remove(class_to_cell[name])
                class_to_cell[name] = source
                cell_order.append(source)
                break   # a cell is already matched, no need to check other class names

    if not cell_order:
        raise ValueError("No matching class definitions found.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n")
        f.write("\n\n".join(cell_order))

    print(f">>> Exported {len(cell_order)} unique cell(s) → {output_path}")

In [228]:
export_model_from_notebook(path_cfg.NOTEBOOK_PATH, path_cfg.MODEL_SCRIPT)

>>> Exported 5 unique cell(s) → C:\Users\98922\Documents\python_scripts\AI\research-notebooks\Quora Questions Pairs\research\model_architecture.py


In [229]:
model = QuoraSiameseClassifier(
    vocab_size=token_cfg.VOCAB_SIZE,
    config=model_cfg,
    embedding=embedding,
    stop_mask=stop_mask
).to(sys_cfg.DEVICE)
print(model)

Glove copied in Embedding Layer...
QuoraSiameseClassifier(
  (embedding): Embedding(20000, 100)
  (emb_norm): Identity()
  (emb_dropout): Dropout(p=0.0, inplace=False)
  (enhc_lstm): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (enhc_lstm_norm): Identity()
  (cross_attention): MultiHeadCrossAttention(
    (heads): ModuleList(
      (0-3): 4 x CrossAttentionHead(
        (W_q): Linear(in_features=512, out_features=128, bias=True)
        (W_k): Linear(in_features=512, out_features=128, bias=True)
        (W_v): Linear(in_features=512, out_features=128, bias=True)
        (V): Linear(in_features=128, out_features=1, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (out_linear): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (proj): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2

In [230]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total Number of Parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable Number of Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Total Number of Parameters: 7333761
Trainable Number of Parameters: 5333761


In [62]:
class EarlyStopping:
    def __init__(self, patience=5, mode="min", min_delta=0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta

        self.should_stop = False
        self.best_score = None
        self.counter = 0

    def step(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return True

        if self.mode == "min":
            improvement = self.best_score - current_score > self.min_delta
        else:
            improvement = current_score - self.best_score > self.min_delta

        if improvement:
            self.best_score = current_score
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

In [63]:
class TrainingHistory:
    def __init__(self):
        self.history = defaultdict(list)

    def update(self, train_loss, val_loss, train_metrics, val_metrics, optimizer):
        self.history["train_loss"].append(train_loss)
        self.history["val_loss"].append(val_loss)
        for k_t, v_t in train_metrics.items():
            self.history[f"train_{k_t.lower()}"].append(v_t)
        for k_v, v_v in val_metrics.items():
            self.history[f"val_{k_v.lower()}"].append(v_v)
        self.history["lr"].append(optimizer.param_groups[0]["lr"])

    def save(self, path: str):
        """Save training history to a JSON file."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)

        print(f"Training History saved successfully at {path}")

    @classmethod
    def load(cls, path: str):
        "Load training history from a JSON file."
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"No file Found at {path}")
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = cls()
        obj.history = data

        return obj

In [64]:
class MLflowTracker:
    def __init__(self, project_name, run_type, config_dict, mlflow_dir, tracking_uri=None):
        model_type = config_dict["model"]["model_type"]
        attention_type = config_dict["model"]["attention_type"]
        self.experiment_name = f"{project_name}/{model_type}"
        self.attn_type = attention_type
        self.model_type = model_type
        self.base_run_name = f"{model_type}-{attention_type}"
        self.run_type = run_type
        self.config_dict = config_dict
        if tracking_uri:
            mlflow.set_tracking_uri(tracking_uri)
        else:
            mlflow.set_tracking_uri("http://localhost:5000")

        self.experiment = mlflow.get_experiment_by_name(self.experiment_name)
        artifact_dir = (mlflow_dir / project_name / model_type).as_uri()
        if self.experiment is None:
            mlflow.create_experiment(
                name=self.experiment_name,
                artifact_location=artifact_dir
            )
            print(f">>> Created new experiment: {self.experiment_name}")
        else:
            print(f">>> Using existing experiment: {self.experiment_name}")

        mlflow.set_experiment(self.experiment_name)

        self.run_name = self._generated_versioned_run_name()
        print(f">>> Run Name: {self.run_name}")

    def _generated_versioned_run_name(self):
        base = self.base_run_name
        experiment = self.experiment
        if self.experiment is None:
            return f"{base}-v1"
    
        runs_df = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string="tags.status = 'completed'"
        )
        if runs_df.empty:
            return f"{base}-v1"
    
        prefix = f"{base}-v"
        mask = runs_df["tags.mlflow.runName"].str.startswith(prefix, na=False)
        matching = runs_df.loc[mask, "tags.mlflow.runName"]
    
        if matching.empty:
            return f"{base}-v1"
    
        versions = []
        for name in matching:
            try:
                version_str = name.split("-v")[-1]
                versions.append(int(version_str))
            except (ValueError, IndexError):
                continue
        next_version = max(versions) + 1 if versions else 1
        return f"{base}-v{next_version}"
            
        
    def start_run(self):
        self.run = mlflow.start_run(run_name=self.run_name)
        self.run_id = self.run.info.run_id
        return self.run

    def log_param(self, param_name, param):
        mlflow.log_param(param_name, param)
    def log_params(self, params):
        mlflow.log_params(params)
    def log_metric(self, name, value, epoch):
        mlflow.log_metric(name, value, step=epoch)
    def log_config_params(self):
        for key, value in self.config_dict.items():
            if key not in ["system", "path"]:
                mlflow.log_params(value) 
        
    def _log_losses(self, train_loss, val_loss, epoch):
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
    def _log_metrics(self, metrics_dict, epoch, prefix):
        for metric, value in metrics_dict.items():
            mlflow.log_metric(f"{prefix}_{metric.lower()}", value, step=epoch)
    def log_epoch(self, epoch, train_loss, val_loss, train_results, val_results, lr):
        self._log_losses(train_loss, val_loss, epoch)
        self._log_metrics(train_results, epoch, prefix="train")
        self._log_metrics(val_results, epoch, prefix="val")
        mlflow.log_metric("learning_rate", lr, step=epoch)
    def save_state_dict(self, state_dict, checkpoint_path):
        mlflow.pytorch.save_state_dict(
            state_dict,
            path=checkpoint_path
        )

    def load_state_dict(self, model, checkpoint_path):
        loaded_state_dict = mlflow.pytorch.load_state_dict(
            checkpoint_path
        )
        model.load_state_dict(loaded_state_dict)
        return model
    def log_best_model(self, model):
        mlflow_logger = logging.getLogger("mlflow.pytorch")
        original_level = mlflow_logger.level
        mlflow_logger.setLevel(logging.ERROR)

        mlflow.pytorch.log_model(
            model,
            name="best_models",
            registered_model_name=self.run_name
        )

        mlflow_logger.setLevel(original_level)
    def log_artifact(self, artifact_path):
        mlflow.log_artifact(artifact_path)

    def log_artifact_folder(self, artifact_folder):
        mlflow.log_artifacts(artifact_folder)

    def build_run_summary(self, best_threshold, training_metrics, calibrated_metrics, total_time, avg_time_per_epoch):
        summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "run_type": self.run_type,
            "model_type": self.model_type,
            "attention_type": self.attn_type,
            "run_id": self.run_id,
            "best_metrics_in_training": training_metrics,
            "best_threshold": best_threshold,
            "best_calibrated_metrics": calibrated_metrics,
            "total_training_time_in_min": total_time,
            "average_training_time_per_epoch": avg_time_per_epoch,
            "params": {}
        }
        for k, v in self.config_dict.items():
            if k not in ["system", "path"]:
                summary["params"][k] = v

        return summary
    def log_summary(self, summary, artifact_name="run_sammary.json"):
        mlflow.log_dict(summary, artifact_name)

    def log_history(self, history, artifact_name="training_history.json"):
        mlflow.log_dict(history, artifact_name)

    def set_final_tags(self, best_score, best_calib_score, best_threshold, total_training_time):
        mlflow.set_tags({
            "status": "completed",
            "model_type": self.model_type,
            "attention_type": self.attn_type,
            "best_training_score": best_score,
            "best_calibrated_score": best_calib_score,
            "best_threshold": best_threshold,
            "total_training_time_in_min": total_training_time
        })

In [65]:
class Trainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        device,
        history,
        mlflow_tracker,
        config=train_cfg,
        scaler=None
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.history = history
        self.config = config
        self.criterion = criterion
        self.criterion_eval = nn.BCEWithLogitsLoss()
        self.optimizer = optimizer
        self.tracker = mlflow_tracker
        self.scaler = scaler
        
        self.early_stopper = EarlyStopping(
            patience=config.EARLY_STOP_PATIENCE,
            mode=config.EARLY_STOP_MODE,
            min_delta=config.EARLY_STOP_MIN_DELTA
        )
        self.scheduler = self._create_scheduler(config.SCHEDULER_TYPE)

        threshold = config.METRICS_THRESHOLD
        self.best_threshold = threshold
        metrics_cls = [BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score]
        self.train_metrics = {m.__name__.replace("Binary", ""): m(threshold).to(device) for m in metrics_cls}
        self.train_metrics["AUROC"] = AUROC(task="binary").to(device)
        self.train_metrics["AveragePrecision"] = AveragePrecision(task="binary").to(device)
        self.val_metrics = {m.__name__.replace("Binary", ""): m(threshold).to(device) for m in metrics_cls}
        self.val_metrics['AUROC'] = AUROC(task="binary").to(device)
        self.val_metrics['AveragePrecision'] = AveragePrecision(task="binary").to(device)
        self.pr_curve = PrecisionRecallCurve(task="binary").to(device)

        self.best_checkpoint_score = float('-inf') if config.CHECKPOINT_MODE == 'max' else float('inf')
        self.checkpoint_mode = config.CHECKPOINT_MODE

        self.train_start = Event(enable_timing=True)
        self.train_end = Event(enable_timing=True)
        self.epoch_start = Event(enable_timing=True)
        self.epoch_end = Event(enable_timing=True)
        self.epoch_durations = []

        self.current_epoch = 0
        
    def _create_scheduler(self, scheduler_type):
        if scheduler_type == "CosineAnnealing":
            return CosineAnnealingLR(
                optimizer=self.optimizer,
                T_max=self.config.EPOCHS,
                eta_min=self.config.SCHEDULER_ETA_MIN
            )
        elif scheduler_type == "ReduceLROnPlateau":
            return ReduceLROnPlateau(
                optimizer=self.optimizer,
                patience=self.config.SCHEDULER_PATIENCE,
                factor=self.config.SCHEDULER_FACTOR,
                mode=self.config.SCHEDULER_MODE,
                min_lr=self.config.SCHEDULER_MIN_LR,
                threshold=self.config.SCHEDULER_THRESHOLD,
                threshold_mode=self.config.SCHEDULER_THRESHOLD_MODE
            )
        elif scheduler_type == "OneCycleLR":
            return OneCycleLR(
                optimizer=self.optimizer,
                max_lr=self.config.LEARNING_RATE,
                epochs=self.config.EPOCHS,
                steps_per_epoch=len(self.train_loader),
                pct_start=self.config.SCHEDULER_PCT_START,        
                div_factor=self.config.SCHEDULER_DIV_FACTOR,        
                final_div_factor=self.config.SCHEDULER_FINAL_DIV_FACTOR 
            )
        elif scheduler_type == "CosineAnnealingWarmRestarts":
            return CosineAnnealingWarmRestarts(
                optimizer=self.optimizer,
                T_0=self.config.SCHEDULER_T_0,
                T_mult=self.config.SCHEDULER_T_MULT,
                eta_min=self.config.SCHEDULER_ETA_MIN
            )
        elif scheduler_type in (None, "none"):
            print("Warning: No scheduler")
            return None
        else:
            raise ValueError(f"Unknown scheduler: {scheduler_type}")

    def _check_scheduler(self, scheduler_value):
        if self.scheduler:
            if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                old_lr = self.optimizer.param_groups[0]["lr"]
                self.scheduler.step(scheduler_value)
                new_lr = self.optimizer.param_groups[0]["lr"]
                if new_lr < old_lr:
                    print(f">>> LR reduced: {old_lr:.6f} → {new_lr:.6f}")
            else:
                self.scheduler.step()
    
    def _unfreeze_embedding(self):
        self.model.embedding.weight.requires_grad = True
        print(">>> Embedding unfrozen (LR unchanged)")

    def _reset_metrics(self, metrics_dict):
        for metric in metrics_dict.values():
            metric.reset()
    
    def _update_metrics(self, metrics_dict, probs, labels):
        for metric in metrics_dict.values():
            metric.update(probs, labels)
    def _update_threshold(self, metrics_dict):
        for metric in metrics_dict.values():
            if hasattr(metric, "threshold"):
                metric.threshold = self.best_threshold
    
    def _compute_metrics(self, metrics_dict):
        return {k:m.compute().item() for k, m in metrics_dict.items()}

    def _get_metric_value(self, metric_name, val_loss, val_results):
        if metric_name == "loss":
            return val_loss
        else:
            return val_results[metric_name]

    def _is_checkpoint_better(self, current_value: float) -> bool:
        if self.checkpoint_mode == 'max':
            return current_value > self.best_checkpoint_score
        else:
            return current_value < self.best_checkpoint_score

    def _backprop_with_scaler(self, q1, q2, labels):
        # Determine if AMP is actually used (only for CUDA)
        use_amp = (self.device.type == 'cuda') and (self.scaler is not None)
        
        with autocast(enabled=use_amp, device_type=self.device.type):
            logits = self.model(q1, q2)
            loss = self.criterion(logits, labels)
        
        if use_amp:
            self.scaler.scale(loss).backward()
            # Unscale before clipping
            self.scaler.unscale_(self.optimizer)
            if self.config.CLIP_NORM is not None and self.config.CLIP_NORM > 0:
                clip_grad_norm_(self.model.parameters(), self.config.CLIP_NORM)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            if self.config.CLIP_NORM is not None and self.config.CLIP_NORM > 0:
                clip_grad_norm_(self.model.parameters(), self.config.CLIP_NORM)
            self.optimizer.step()
        
        return loss, logits
            
    def train_one_epoch(self):
        self.model.train()
        self._update_threshold(self.train_metrics)
        self._reset_metrics(self.train_metrics)
        total_loss = 0.0

        for batch in tqdm(self.train_loader, desc="Train", leave=True):
            q1 = batch["q1"].to(self.device)
            q2 = batch["q2"].to(self.device)
            labels = batch["label"].to(self.device).long()

            self.optimizer.zero_grad()
            
            loss, logits = self._backprop_with_scaler(q1, q2, labels)
            
            total_loss += (loss.item() * q1.size(0))

            probs = torch.sigmoid(logits)
            self._update_metrics(self.train_metrics, probs, labels)

        total_loss /= len(self.train_loader.dataset)
        results = self._compute_metrics(self.train_metrics)
        return total_loss, results

    @torch.no_grad()
    def evaluate(self, use_optimal_threshold=True):
        self.model.eval()
        total_loss = 0.0
        all_logits, all_labels = [], []
    
        for batch in tqdm(self.val_loader, desc="Validation", leave=True):
            q1 = batch["q1"].to(self.device)
            q2 = batch["q2"].to(self.device)
            labels = batch["label"].to(self.device).long()
    
            logits = self.model(q1, q2)
            loss = self.criterion_eval(logits, labels.float())
            total_loss += loss.item() * q1.size(0)
    
            all_logits.append(logits)
            all_labels.append(labels)
    
        total_loss /= len(self.val_loader.dataset)
    
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        all_probs = torch.sigmoid(all_logits)
    
        # Find optimal threshold using self.pr_curve (if use_optimal_threshold)
        if use_optimal_threshold:
            self.pr_curve.reset()
            all_labels = all_labels.long()
            self.pr_curve.update(all_probs, all_labels)
            precision, recall, thresholds = self.pr_curve.compute()
            # Ensure thresholds is not empty
            if thresholds.numel() > 0:
                f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
                best_idx = torch.argmax(f1_scores)
                self.best_threshold = thresholds[best_idx].item()
            else:
                self.best_threshold = self.config.METRICS_THRESHOLD
        else:
            self.best_threshold = self.config.METRICS_THRESHOLD
    
        self._update_threshold(self.val_metrics)
    
        # Reset, update, compute using existing helpers
        self._reset_metrics(self.val_metrics)
        self._update_metrics(self.val_metrics, all_probs, all_labels)
        results = self._compute_metrics(self.val_metrics)
    
        return total_loss, results
    def log_one_epoch(self, train_loss, train_results, val_loss, val_results, best_thresh, lr):
        print(
            f"Training Results:\n\tLoss --> {train_loss:.4f}"
        )
        train_string = ""
        val_string = ""
        for k, v in train_results.items():
            train_result = f"{k} --> {v:.4f} | "
            train_string += train_result
        print(f"\t{train_string}")
        print(f"\tLearning Rate --> {lr:.4f}\n")
        print(
            f"Validation Results:\n\tLoss --> {val_loss:.4f}"
            )
        print(f"\tOptimal Best Threhsold --> {best_thresh}")
        for k, v in val_results.items():
            val_result = f"{k} --> {v:.4f} | "
            val_string += val_result
        print(f"\t{val_string}")

    @torch.no_grad()
    def find_optimal_threshold(self):
        self.model.eval()
        _, results = self.evaluate(use_optimal_threshold=True)
        best_threshold = self.best_threshold
    
        print(f">>> Optimal threshold: {best_threshold:.4f}")
        print(f">>> At that threshold --> F1: {results['F1Score']:.4f}, Precision: {results['Precision']:.4f}, Recall: {results['Recall']:.4f}")
        return best_threshold, results
    
    def fit(self, num_epochs, config=path_cfg):
        self.train_start.record()
        
        print(">>> Training Started...")
        with self.tracker.start_run() as run:
            self.tracker.log_config_params()
            self.tracker.log_artifact(config.VOCABS_PATH)
            self.tracker.log_artifact(config.LABEL_MAPPING_PATH)
            self.tracker.log_artifact(config.CONFIG_PATH)
            self.tracker.log_artifact(config.MODEL_SCRIPT)
            self.tracker.log_artifact(config.REQ_SCRIPT)
            self.tracker.log_artifact(config.REQ_TEXT)

            best_metrics = {}
            
            for epoch in range(num_epochs):
                self.current_epoch = epoch
                print(f"Epoch {epoch+1}/{num_epochs}")
                self.epoch_start.record()
                torch.cuda.synchronize()
                
                if self.config.UNFREEZE_EPOCH is not None and epoch == self.config.UNFREEZE_EPOCH:
                    print(f">>> Embedding is unfrozen from epoch {epoch+1}")
                    self._unfreeze_embedding()
                
                train_loss, train_results = self.train_one_epoch()
                val_loss, val_results = self.evaluate()

                early_stop_value = self._get_metric_value(
                    self.config.EARLY_STOP_METRIC, val_loss, val_results
                )
                scheduler_value = self._get_metric_value(
                    self.config.SCHEDULER_METRIC, val_loss, val_results
                )
                checkpoint_value = self._get_metric_value(
                    self.config.CHECKPOINT_METRIC, val_loss, val_results
                )
        
                self._check_scheduler(scheduler_value)
                    
                self.history.update(
                    train_loss=train_loss,
                    val_loss=val_loss,
                    train_metrics=train_results,
                    val_metrics=val_results,
                    optimizer=self.optimizer
                )
        
                lr = self.optimizer.param_groups[0]["lr"]
                self.log_one_epoch(
                    train_loss=train_loss,
                    train_results=train_results,
                    val_loss=val_loss,
                    val_results=val_results,
                    best_thresh=self.best_threshold,
                    lr=lr
                )
                self.tracker.log_epoch(
                    epoch=epoch,
                    train_loss=train_loss,
                    val_loss=val_loss,
                    train_results=train_results,
                    val_results=val_results,
                    lr=lr
                )
                self.tracker.log_metric("best_threshold", self.best_threshold, epoch)
                self.early_stopper.step(early_stop_value)
                is_better_checkpoint = False
                if self._is_checkpoint_better(checkpoint_value):
                    self.best_checkpoint_score = checkpoint_value 

                    best_metrics = val_results
                    self.tracker.save_state_dict(self.model.state_dict(), config.CHECKPOINT_DIR)
                    print(f">>> Best model saved! ({self.config.CHECKPOINT_METRIC} --> {checkpoint_value:.4f})")
                    
                if self.early_stopper.should_stop:
                    print(f">>> Early stopping triggered at Epoch {epoch+1}")
                    break

                self.epoch_end.record()
                torch.cuda.synchronize()
                
                epoch_duration = self.epoch_start.elapsed_time(self.epoch_end) / 1000
                self.epoch_durations.append(epoch_duration)
                print("="*sys_cfg.NEXT_LINE_COUNTER)


            self.train_end.record()
            torch.cuda.synchronize()
            total_training_time = self.train_start.elapsed_time(self.train_end) / 1000
            avg_time_per_epoch = sum(self.epoch_durations) / len(self.epoch_durations)
            total_training_time_in_min = round(total_training_time/60, 2)
            avg_time_per_epoch_in_min = round(avg_time_per_epoch/60, 2)

            self.model = self.tracker.load_state_dict(self.model, config.CHECKPOINT_DIR)
            self.model.eval()
            final_best_threshold, calibrated_results = self.find_optimal_threshold()
            self.tracker.log_artifact_folder(config.CHECKPOINT_DIR)
            self.tracker.log_best_model(self.model)
            print(">>> The Best Model registered at MLflow successfully!")
            summary = self.tracker.build_run_summary(
                best_threshold=final_best_threshold,
                training_metrics=best_metrics,
                calibrated_metrics=calibrated_results,
                total_time=total_training_time_in_min,
                avg_time_per_epoch=avg_time_per_epoch_in_min
            )
            self.tracker.log_param(
                f"best_training_{self.config.CHECKPOINT_METRIC}", 
                self.best_checkpoint_score
            )
            self.tracker.log_param(
                f"best_calibrated_{self.config.CHECKPOINT_METRIC}", 
                calibrated_results[self.config.CHECKPOINT_METRIC]
            )
            self.tracker.log_param("best_threshold", final_best_threshold)
            self.tracker.log_params(calibrated_results)
            self.tracker.log_param("total_training_time_in_min", total_training_time_in_min)
            self.tracker.log_param("average_time_per_epoch_in_min", avg_time_per_epoch_in_min)
            self.tracker.log_summary(summary)
            self.tracker.log_history(self.history)
            self.tracker.log_artifact(config.NOTEBOOK_PATH)
            self.tracker.set_final_tags(
                best_score=self.best_checkpoint_score,
                best_calib_score=calibrated_results[self.config.CHECKPOINT_METRIC],
                best_threshold=final_best_threshold,
                total_training_time=total_training_time_in_min
            )

In [66]:
class BCEWithLabelSmoothing(nn.Module):
    def __init__(self,  epsilon, reduction="mean"):
        super().__init__()
        self.epsilon = epsilon
        self.criterion = nn.BCEWithLogitsLoss(reduction=reduction)

    def forward(self, logits, labels):
        labels = labels.float()
        labels = labels * (1 - self.epsilon) + (1 - labels) * self.epsilon
        return self.criterion(logits, labels)

In [68]:
mlflow_tracker = MLflowTracker(
    project_name="Quora-Question-Pairs",
    run_type="exploring-best-architecture",
    config_dict=configs,
    mlflow_dir=path_cfg.MLFLOW_DIR,
    tracking_uri=sys_cfg.MLFLOW_TRACKING_URI
)
history = TrainingHistory()
criterion = BCEWithLabelSmoothing(epsilon=model_cfg.LABEL_SMOOTHING)
optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'embedding' not in n], 
     'lr': train_cfg.LEARNING_RATE},
    {'params': [model.embedding.weight], 
     'lr': train_cfg.LEARNING_RATE / 10, 
     'requires_grad': False}
], weight_decay=train_cfg.WEIGHT_DECAY)

>>> Using existing experiment: Quora-Question-Pairs/LSTM_attention
>>> Run Name: LSTM_attention-MultiHead-CrossAttention-Bahdanau-v6


In [69]:
trainer = Trainer(
    model=model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    device=sys_cfg.DEVICE,
    history=history,
    config=train_cfg,
    mlflow_tracker=mlflow_tracker,
    scaler=scaler
)

In [70]:
trainer.fit(
    num_epochs=train_cfg.EPOCHS,
    config=path_cfg,
)

>>> Training Started...
Epoch 1/50


Train:   0%|          | 0/2843 [00:00<?, ?it/s]

C:\Users\98922\ai-env\Lib\site-packages\torch\nn\modules\conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1028.)
  return F.conv1d(


Validation:   0%|          | 0/316 [00:00<?, ?it/s]

Training Results:
	Loss --> 0.5534
	Accuracy --> 0.7405 | Precision --> 0.6769 | Recall --> 0.5687 | F1Score --> 0.6181 | AUROC --> 0.8038 | AveragePrecision --> 0.7024 | 
	Learning Rate --> 0.0003

Validation Results:
	Loss --> 0.4613
	Optimal Best Threhsold --> 0.3883788287639618
	Accuracy --> 0.7668 | Precision --> 0.6479 | Recall --> 0.8070 | F1Score --> 0.7188 | AUROC --> 0.8570 | AveragePrecision --> 0.7767 | 
>>> Best model saved! (F1Score --> 0.7188)
Epoch 2/50


...

Training Results:
	Loss --> 0.3531
	Accuracy --> 0.9066 | Precision --> 0.8717 | Recall --> 0.8760 | F1Score --> 0.8738 | AUROC --> 0.9619 | AveragePrecision --> 0.9339 | 
	Learning Rate --> 0.0003

Validation Results:
	Loss --> 0.3756
	Optimal Best Threhsold --> 0.4758058786392212
	Accuracy --> 0.8417 | Precision --> 0.7576 | Recall --> 0.8399 | F1Score --> 0.7966 | AUROC --> 0.9155 | AveragePrecision --> 0.8689 | 
>>> Best model saved! (F1Score --> 0.7966)
>>> Early stopping triggered at Epoch 13


Validation:   0%|          | 0/316 [00:00<?, ?it/s]

>>> Optimal threshold: 0.4758
>>> At that threshold --> F1: 0.7966, Precision: 0.7576, Recall: 0.8399


Successfully registered model 'LSTM_attention-MultiHead-CrossAttention-Bahdanau-v6'.
Created version '1' of model 'LSTM_attention-MultiHead-CrossAttention-Bahdanau-v6'.


>>> The Best Model registered at MLflow successfully!
